In [1]:
!pip install triton

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.1 MB/s eta 0:00:00:00:0100:01


In [2]:
import torch
import triton
import triton.language as tl


In [ ]:
def naive_softmax(x):
    """eager mode softmax"""
    x_max = torch.max(x,dim=1)[0]
    safe_x = x - x_max[:, None]
    numerator = torch.exp(safe_x)
    denominator = torch.sum(numerator, dim=1)
    sm_out = numerator/denominator[:,None]
    return sm_out
       
eager_out = naive_softmax(sample)
print(eager_out)

In [ ]:
@triton.jit
def _softmax_fwd_kernel(
    output_ptr,
    stride_output_row,
    input_ptr,
    stride_input_row,
    num_cols,
    block_size: tl.constexpr,
):
    #setup our input pts befre we start
    row_index = tl.program_id(0)
    row_start = input_ptr + (row_index* stride_input_row)
    
    

#driver program
def softmax(x):
    rows, cols = x.shape
    assert x.dim() == 2, f"only 2d tensor for now"
    #if 1000 rows, we parallelize row size 
    block_size = triton.next_power_of_2(cols) 
    # if we have 1000 cols = then block size becomes 1024
    num_warps = 4 #4x32 threads 
    if block_size > 2047:
        num_warps = 8
    if block_size > 4095:
        num_warps = 16
    
    grid = (rows,)
    #allocate output buffer
    
    sm_out = torch.empty_like(x)
    _softmax_fwd_kernel[grid](
        sm_out, 
        sm_out.stride(0), 
        x, 
        x.stride(0),
        cols,
        block_size = block_size,
        num_warps = num_warps
        )
    return sm_out
    


batch size x num heads x (seqlen/batch size q)

In [ ]:
@triton.jit
def flash_fwd_kernel(
    Q_ptr, K_ptr, V_ptr, #these are pointers to Q,K,V
    O_ptr, L_ptr, #ptrs to output
    stride_qb, stride_qq, stride_qd,
    stride_kb, stride_kk, stride_kd,
    stride_vb, stride_vk, stride_vd,
    stride_ob, stride_oq, stride_od,
    stride_lb, stride_lq,
    N_QUERIES, N_KEYS,
    scale,
    D: tl.constexpr,
    Q_TILE_SIZE: tl.constexpr,
    K_TILE_SIZE: tl.constexpr,
):
    query_tile_index = tl.program_id(0)
    batch_index = tl.program_id(1)
    
    Q_block_ptr = tl.make_block_ptr(
        Q_ptr + batch_index * stride_qb,
        shape=(N_QUERIES, D),
        strides=(stride_qq, stride_qd),
        offsets=(query_tile_index * Q_TILE_SIZE, 0),
        block_shape=(Q_TILE_SIZE, D),
        order=(1, 0),
    )
    
    K_block_ptr = tl.make_block_ptr(
        K_ptr + batch_index * stride_kb,
        shape=(N_KEYS, D),
        strides=(stride_kk, stride_kd),
        offsets=(0, 0),
        block_shape=(K_TILE_SIZE, D),
        order=(1, 0),
    )
    
    V_block_ptr = tl.make_block_ptr(
        V_ptr + batch_index * stride_vb,
        shape=(N_KEYS, D),
        strides=(stride_vk, stride_vd),
        offsets=(0, 0),
        block_shape=(K_TILE_SIZE, D),
        order=(1, 0),
    )
    
    # Load Q tile -- stays in SRAM for the entire loop
    Qi = tl.load(Q_block_ptr)  # (Q_TILE_SIZE, D)
    
    # Initialize accumulators
    mi = tl.full((Q_TILE_SIZE,), float('-inf'), dtype=tl.float32)  # row-wise max
    li = tl.zeros((Q_TILE_SIZE,), dtype=tl.float32)                # row-wise sum
    Oi = tl.zeros((Q_TILE_SIZE, D), dtype=tl.float32)              # output accumulator
    
    # Loop over K/V tiles
    num_k_tiles = tl.cdiv(N_KEYS, K_TILE_SIZE)
    for _ in range(num_k_tiles):
        Kj = tl.load(K_block_ptr)  # (K_TILE_SIZE, D)
        Vj = tl.load(V_block_ptr)  # (K_TILE_SIZE, D)
        
        # Compute attention scores: (Q_TILE_SIZE, K_TILE_SIZE)
        Sij = tl.dot(Qi, tl.trans(Kj)) * scale
        
        # New row-wise max
        mi_new = tl.maximum(mi, tl.max(Sij, axis=1))
        
        # Correction factor for old accumulators
        alpha = tl.exp(mi - mi_new)
        
        # Exponentiate scores with new max
        Pij = tl.exp(Sij - mi_new[:, None])
        
        # Update running sum and output
        li = alpha * li + tl.sum(Pij, axis=1)
        Oi = alpha[:, None] * Oi + tl.dot(Pij.to(Vj.dtype), Vj)
        
        mi = mi_new
        
        # Advance K and V block pointers to next tile
        K_block_ptr = tl.advance(K_block_ptr, (K_TILE_SIZE, 0))
        V_block_ptr = tl.advance(V_block_ptr, (K_TILE_SIZE, 0))
    
    # Normalize output
    Oi = Oi / li[:, None]
    Li = mi + tl.log(li)
    
    # Store output
    O_block_ptr = tl.make_block_ptr(
        O_ptr + batch_index * stride_ob,
        shape=(N_QUERIES, D),
        strides=(stride_oq, stride_od),
        offsets=(query_tile_index * Q_TILE_SIZE, 0),
        block_shape=(Q_TILE_SIZE, D),
        order=(1, 0),
    )
    tl.store(O_block_ptr, Oi.to(Q_ptr.dtype.element_ty))
    
    # Store log-sum-exp
    L_ptrs = L_ptr + batch_index * stride_lb + query_tile_index * Q_TILE_SIZE + tl.arange(0, Q_TILE_SIZE)
    tl.store(L_ptrs, Li)

In [ ]:
class FlashAttention2(torch.autograd.Function):
    @staticmethod
    def forward(ctx, Q, K, V, is_causal=False):
        B, Nq, d = Q.shape
        _, Nk, _ = K.shape
        
        scale = 1.0 / (d ** 0.5)
        
        Q_TILE_SIZE = 64
        K_TILE_SIZE = 64
        
        # Allocate output buffers
        O = torch.empty_like(Q)
        L = torch.empty(B, Nq, device=Q.device, dtype=torch.float32)
        
        # Create the grid: one program per query tile per batch element
        grid = (triton.cdiv(Nq, Q_TILE_SIZE), B)
        
        # Call the kernel
        flash_fwd_kernel[grid](
            Q, K, V,
            O, L,
            Q.stride(0), Q.stride(1), Q.stride(2),
            K.stride(0), K.stride(1), K.stride(2),
            V.stride(0), V.stride(1), V.stride(2),
            O.stride(0), O.stride(1), O.stride(2),
            L.stride(0), L.stride(1),
            Nq, Nk,
            scale,
            D=d,
            Q_TILE_SIZE=Q_TILE_SIZE,
            K_TILE_SIZE=K_TILE_SIZE,
        )
        
        ctx.save_for_backward(L, Q, K, V, O)
        return O
    
    @staticmethod
    def backward(ctx, *grad_outputs):
        raise NotImplementedError